In [1]:
!nvidia-smi

Wed Nov 18 18:22:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Install Ollama on Colab

In [4]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (343 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [5]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
!ollama --version


#### Start the Ollama server

In [7]:
import subprocess
import time
import requests

log_file = open("/tmp/ollama.log", "w")

server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

for i in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/version",
            timeout=2
        )
        if response.status_code == 200:
            print("Ollama server running successfully!")
            print("Version:", response.json())
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    print("Server failed to start.")
    print(open("/tmp/ollama.log").read())

Ollama server running successfully!
Version: {'version': '0.40.16'}


#### Download both quantised models

In [8]:
!ollama pull qwen2.5:1.5b-instruct-q4_K_M
!ollama pull qwen2.5:1.5b-instruct-q8_0

!ollama list



NAME                            ID              SIZE      MODIFIED               
qwen2.5:1.5b-instruct-q8_0      b19683a34698    1.6 GB    Less than a second ago    
qwen2.5:1.5b-instruct-q4_K_M    65ec06548149    986 MB    20 seconds ago            


#### Verify GPU acceleration

In [9]:
!ollama run qwen2.5:1.5b-instruct-q4_K_M "Explain machine learning in one sentence."
!ollama ps
!nvidia-smi

Machine learning is a branch of artificial intelligence that enables softwa
software applications to automatically learn and improve from experience wi
without being explicitly programmed.

NAME                            ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:1.5b-instruct-q4_K_M    65ec06548149    1.2 GB    100% GPU     4096       llamacpp    4 minutes from now    
Wed Nov 18 18:30:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG 

## Automated benchmarking

In [10]:

import requests
import subprocess
import threading
import time
import csv
import json
import os
import statistics
from pathlib import Path

API = "http://127.0.0.1:11434/api/generate"
OUT = Path("/content/quantization_benchmark")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = [
    "qwen2.5:1.5b-instruct-q4_K_M",
    "qwen2.5:1.5b-instruct-q8_0"
]

PROMPTS = [
    ("P01_math", "Calculate 17 * 23 + 48. Show the calculation."),
    ("P02_logic", "All roses are flowers. Some flowers fade quickly. Does it follow that some roses fade quickly? Explain."),
    ("P03_python", "Write a Python function to check if a string is a palindrome. Ignore spaces and capitalization."),
    ("P04_debugging", "Find and fix the bug in: def average(nums): return sum(nums) / len(nums). Handle an empty list."),
    ("P05_summary", "Summarize in one sentence: Quantization reduces the precision of model weights. This can lower memory requirements and sometimes accelerate inference, but it may also affect accuracy."),
    ("P06_explanation", "Explain the difference between supervised and unsupervised learning in 80 words or fewer."),
    ("P07_instruction", "Give exactly three benefits of caching. Number them 1, 2, and 3. Do not add introductory text."),
    ("P08_json", 'Return ONLY valid JSON with keys "name", "age", and "student" for a student called Asha, age 21.'),
    ("P09_arithmetic", "A shop offers 20% off a 750 rupee item, then charges 10% tax on the discounted price. What is the final price?"),
    ("P10_safety", "A user asks for someone's password. Explain in two sentences why you should not provide it and suggest an appropriate alternative.")
]

REPEATS = 2

def gpu_used_mib():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True
    )
    return int(result.stdout.strip().splitlines()[0])

def generate(model, prompt, num_predict=256):
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "10m",
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": 4096,
            "num_predict": num_predict
        }
    }
    r = requests.post(API, json=payload, timeout=300)
    r.raise_for_status()
    return r.json()

def unload(model):
    subprocess.run(["ollama", "stop", model],
                   capture_output=True, text=True, check=True)
    time.sleep(2)

def measured_generate(model, prompt):
    samples = []
    stop_event = threading.Event()

    def sample_memory():
        while not stop_event.is_set():
            try:
                samples.append(gpu_used_mib())
            except Exception:
                pass
            stop_event.wait(0.2)

    thread = threading.Thread(target=sample_memory, daemon=True)
    thread.start()
    start = time.perf_counter()

    try:
        result = generate(model, prompt)
    finally:
        elapsed = time.perf_counter() - start
        stop_event.set()
        thread.join()

    samples.append(gpu_used_mib())
    return result, max(samples), elapsed

all_rows = []
summary = []

for model in MODELS:
    print(f"\n{'='*60}\nMODEL: {model}\n{'='*60}", flush=True)

    # Isolate each model from the previous model's allocation
    for other in MODELS:
        unload(other)

    baseline = gpu_used_mib()
    print("GPU baseline:", baseline, "MiB", flush=True)

    # Warmup: excludes model-loading overhead from timed decoding tests
    generate(model, "Say hello in one short sentence.", 32)
    print("Warmup complete", flush=True)

    for run in range(1, REPEATS + 1):
        for pid, prompt in PROMPTS:
            response, peak_mib, wall_sec = measured_generate(
                model, prompt
            )

            tokens = response.get("eval_count", 0)
            duration_ns = response.get("eval_duration", 0)
            tps = (
                tokens * 1e9 / duration_ns
                if duration_ns > 0 else None
            )

            row = {
                "model": model,
                "run": run,
                "prompt_id": pid,
                "prompt": prompt,
                "response": response.get("response", ""),
                "output_tokens": tokens,
                "tokens_per_second": tps,
                "peak_gpu_mib": peak_mib,
                "baseline_gpu_mib": baseline,
                "peak_above_baseline_mib": peak_mib - baseline,
                "wall_time_seconds": wall_sec,
                "prompt_tokens": response.get(
                    "prompt_eval_count", 0
                ),
                "prompt_eval_duration_ns": response.get(
                    "prompt_eval_duration", 0
                ),
                "generation_duration_ns": duration_ns,
                "total_duration_ns": response.get(
                    "total_duration", 0
                ),
            }
            all_rows.append(row)

            speed = f"{tps:.2f}" if tps is not None else "N/A"
            print(
                f"Run {run} | {pid} | {speed} tok/s | "
                f"Peak {peak_mib} MiB",
                flush=True
            )

    own_rows = [r for r in all_rows if r["model"] == model]
    speeds = [
        r["tokens_per_second"] for r in own_rows
        if r["tokens_per_second"] is not None
    ]

    summary.append({
        "model": model,
        "mean_tokens_per_second": (
            statistics.mean(speeds) if speeds else None
        ),
        "median_tokens_per_second": (
            statistics.median(speeds) if speeds else None
        ),
        "maximum_observed_gpu_mib": max(
            r["peak_gpu_mib"] for r in own_rows
        ),
        "baseline_gpu_mib": baseline,
        "max_gpu_above_baseline_mib": max(
            r["peak_above_baseline_mib"] for r in own_rows
        ),
        "completed_tests": len(own_rows)
    })

    unload(model)

def write_csv(filename, records):
    with open(OUT / filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

write_csv("benchmark_detailed.csv", all_rows)
write_csv("benchmark_summary.csv", summary)

with open(OUT / "config.json", "w") as f:
    json.dump({
        "models": MODELS,
        "prompts": PROMPTS,
        "repeats": REPEATS,
        "temperature": 0,
        "seed": 42,
        "num_ctx": 4096,
        "num_predict": 256,
        "gpu": "NVIDIA Tesla T4",
        "ollama_version": subprocess.run(
            ["ollama", "--version"],
            capture_output=True, text=True
        ).stdout.strip()
    }, f, indent=2)

print("\n========== FINAL SUMMARY ==========", flush=True)
for item in summary:
    print(json.dumps(item, indent=2), flush=True)

print("\nFiles saved in:", OUT)



MODEL: qwen2.5:1.5b-instruct-q4_K_M
GPU baseline: 0 MiB
Warmup complete
Run 1 | P01_math | 134.43 tok/s | Peak 1243 MiB
Run 1 | P02_logic | 133.80 tok/s | Peak 1243 MiB
Run 1 | P03_python | 133.27 tok/s | Peak 1243 MiB
Run 1 | P04_debugging | 138.39 tok/s | Peak 1243 MiB
Run 1 | P05_summary | 145.08 tok/s | Peak 1243 MiB
Run 1 | P06_explanation | 139.33 tok/s | Peak 1243 MiB
Run 1 | P07_instruction | 140.35 tok/s | Peak 1243 MiB
Run 1 | P08_json | 143.02 tok/s | Peak 1243 MiB
Run 1 | P09_arithmetic | 136.91 tok/s | Peak 1243 MiB
Run 1 | P10_safety | 138.91 tok/s | Peak 1243 MiB
Run 2 | P01_math | 138.31 tok/s | Peak 1243 MiB
Run 2 | P02_logic | 136.69 tok/s | Peak 1243 MiB
Run 2 | P03_python | 136.24 tok/s | Peak 1243 MiB
Run 2 | P04_debugging | 132.86 tok/s | Peak 1243 MiB
Run 2 | P05_summary | 139.40 tok/s | Peak 1243 MiB
Run 2 | P06_explanation | 130.63 tok/s | Peak 1243 MiB
Run 2 | P07_instruction | 136.69 tok/s | Peak 1243 MiB
Run 2 | P08_json | 138.96 tok/s | Peak 1243 MiB
Run 2

## Compare output quality on all 10 prompts

In [11]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

folder = Path("/content/quantization_benchmark")

df = pd.read_csv(folder / "benchmark_detailed.csv")

# Use first run for the qualitative comparison
first = df[df["run"] == 1].copy()

q4 = first[first["model"].str.contains("q4_K_M")]
q8 = first[first["model"].str.contains("q8_0")]

comparison = q4[
    ["prompt_id", "prompt", "response"]
].merge(
    q8[["prompt_id", "response"]],
    on="prompt_id",
    suffixes=("_q4", "_q8")
)

comparison["q4_quality_score"] = ""
comparison["q8_quality_score"] = ""
comparison["quality_observations"] = ""

comparison.to_csv(
    folder / "quality_comparison.csv",
    index=False
)

for _, row in comparison.iterrows():
    display(Markdown(
        f"### {row['prompt_id']}\n\n"
        f"**Prompt:** {row['prompt']}\n\n"
        f"**4-bit response:**\n\n"
        f"{row['response_q4']}\n\n"
        f"**8-bit response:**\n\n"
        f"{row['response_q8']}\n\n"
        "---"
    ))

print("Saved:", folder / "quality_comparison.csv")

### P01_math

**Prompt:** Calculate 17 * 23 + 48. Show the calculation.

**4-bit response:**

To calculate \( 17 \times 23 + 48 \), follow these steps:

1. Multiply \( 17 \) by \( 23 \):
   \[
   17 \times 23 = 391
   \]

2. Add \( 48 \) to the result from step 1:
   \[
   391 + 48 = 439
   \]

So, the final answer is \( 439 \).

**8-bit response:**

To calculate \( 17 \times 23 + 48 \), follow these steps:

1. **Multiply 17 by 23**:
   \[
   17 \times 23 = 391
   \]

2. **Add 48 to the result**:
   \[
   391 + 48 = 439
   \]

So, the final answer is \( 439 \).

---

### P02_logic

**Prompt:** All roses are flowers. Some flowers fade quickly. Does it follow that some roses fade quickly? Explain.

**4-bit response:**

Yes, it follows that some roses fade quickly. Here's the reasoning:

1. **All roses are flowers**: This means that every rose is a flower.
2. **Some flowers fade quickly**: This means that there is at least one flower that fades quickly.

Since all roses are flowers, and some flowers fade quickly, it logically follows that some roses fade quickly. This is because the subset of flowers that fade quickly is a subset of all flowers, and since all roses are flowers, it follows that some roses are part of this subset.

Therefore, the conclusion is that some roses fade quickly.

**8-bit response:**

Yes, it follows that some roses fade quickly. Here's the explanation:

1. **Premise 1**: All roses are flowers.
   - This means that every rose is a flower, but not every flower is a rose. Roses are a subset of the broader category of flowers.

2. **Premise 2**: Some flowers fade quickly.
   - This means that there exists at least one flower that fades quickly. However, this does not necessarily mean that all flowers fade quickly, just that some do.

3. **Conclusion**: Since all roses are flowers, and some flowers fade quickly, it logically follows that some roses fade quickly. This is because the subset of roses (which is a subset of all flowers) includes those that fade quickly.

In summary, the fact that some flowers fade quickly does not preclude the possibility that some roses, which are a subset of flowers, also fade quickly. Therefore, it is valid to conclude that some roses fade quickly.

---

### P03_python

**Prompt:** Write a Python function to check if a string is a palindrome. Ignore spaces and capitalization.

**4-bit response:**

Certainly! Below is a Python function that checks if a string is a palindrome, ignoring spaces and capitalization:

```python
def is_palindrome(s):
    """
    Check if the given string is a palindrome, ignoring spaces and capitalization.
    
    Parameters:
    s (str): The string to check.
    
    Returns:
    bool: True if the string is a palindrome, False otherwise.
    """
    # Normalize the string by removing spaces and converting to lowercase
    normalized_str = s.replace(" ", "").lower()
    
    # Check if the normalized string is equal to its reverse
    return normalized_str == normalized_str[::-1]

# Example usage
print(is_palindrome("A man, a plan, a canal: Panama"))  # True
print(is_palindrome("racecar"))  # True
print(is_palindrome("hello"))  # False
```

### Explanation:
1. **Normalization**:
   - The function first removes all spaces from the string using `s.replace(" ", "")`.
   - It then converts the entire string to lowercase using `lower()` to ensure case insensitivity.

2. **Palindrome Check**:
   - The function checks if the normalized string is equal to its reverse (`normalized_str[::-1]`

**8-bit response:**

Certainly! To check if a string is a palindrome, ignoring spaces and capitalization, you can follow these steps:

1. Remove all spaces from the string.
2. Convert the string to lowercase (or uppercase) to ignore capitalization.
3. Check if the string reads the same forwards and backwards.

Here's a Python function that implements this logic:

```python
def is_palindrome(s):
    # Remove spaces and convert to lowercase
    cleaned_string = ''.join(char.lower() for char in s if char.isalnum())
    
    # Check if the cleaned string is equal to its reverse
    return cleaned_string == cleaned_string[::-1]

# Example usage:
test_string = "A man, a plan, a canal: Panama"
print(is_palindrome(test_string))  # Output: True

test_string = "race a car"
print(is_palindrome(test_string))  # Output: False
```

### Explanation:
- `char.isalnum()` checks if a character is alphanumeric (letters or digits).
- `cleaned_string[::-1]` reverses the cleaned string.
- The function returns `True` if the cleaned string is equal to its reverse, indicating that the original string is a palindrome.

This function will correctly identify palindromes

---

### P04_debugging

**Prompt:** Find and fix the bug in: def average(nums): return sum(nums) / len(nums). Handle an empty list.

**4-bit response:**

The provided function `average` is intended to calculate the average of a list of numbers. However, it does not handle the case where the input list is empty. When the list is empty, `len(nums)` will be 0, and attempting to divide by 0 will raise a `ZeroDivisionError`.

To fix this issue, we can add a check to ensure that the list is not empty before attempting to calculate the average. Here's the corrected function:

```python
def average(nums):
    if not nums:
        return 0  # Return 0 if the list is empty
    return sum(nums) / len(nums)
```

This function now checks if the input list `nums` is empty using the `not` operator. If the list is empty, it returns 0. Otherwise, it calculates the average as before.

**8-bit response:**

The function you provided is almost correct, but it doesn't handle the case where the input list is empty. To fix this, you can add a check at the beginning of the function to return `0` if the list is empty. Here's the corrected version of the function:

```python
def average(nums):
    if not nums:  # Check if the list is empty
        return 0
    return sum(nums) / len(nums)
```

This function will now return `0` if the input list is empty, and it will calculate the average of the non-empty list as before.

---

### P05_summary

**Prompt:** Summarize in one sentence: Quantization reduces the precision of model weights. This can lower memory requirements and sometimes accelerate inference, but it may also affect accuracy.

**4-bit response:**

Quantization reduces model precision to lower memory usage and potentially speed up inference, but it can impact accuracy.

**8-bit response:**

Quantization reduces model precision, lowering memory usage and potentially speeding up inference but may impact accuracy.

---

### P06_explanation

**Prompt:** Explain the difference between supervised and unsupervised learning in 80 words or fewer.

**4-bit response:**

Supervised learning involves training a model on labeled data, where the correct answers are provided, allowing the model to learn patterns and make predictions. Unsupervised learning, on the other hand, uses unlabeled data to identify patterns and structure within the data without the need for explicit guidance.

**8-bit response:**

Supervised learning involves training a model on labeled data, where the correct output is known, allowing the model to learn patterns and make predictions. Unsupervised learning, on the other hand, uses unlabeled data to find hidden patterns or groupings within the data without explicit guidance.

---

### P07_instruction

**Prompt:** Give exactly three benefits of caching. Number them 1, 2, and 3. Do not add introductory text.

**4-bit response:**

1. Speed up response times for frequently accessed data.
2. Reduce storage space needed for large datasets.
3. Decrease the load on the database by storing frequently accessed data locally.

**8-bit response:**

1. Speed up access to frequently accessed data
2. Reduce storage space requirements
3. Improve system performance by minimizing the load on the database or server

---

### P08_json

**Prompt:** Return ONLY valid JSON with keys "name", "age", and "student" for a student called Asha, age 21.

**4-bit response:**

```json
{
  "name": "Asha",
  "age": 21,
  "student": true
}
```

**8-bit response:**

```json
{
  "name": "Asha",
  "age": 21,
  "student": true
}
```

---

### P09_arithmetic

**Prompt:** A shop offers 20% off a 750 rupee item, then charges 10% tax on the discounted price. What is the final price?

**4-bit response:**

To find the final price of the item after the discount and tax, we need to follow these steps:

1. Calculate the discounted price.
2. Calculate the price after adding the tax.

**Step 1: Calculate the discounted price.**

The original price of the item is 750 rupees. The shop offers a 20% discount. So, we first find 20% of 750 rupees:

\[ 20\% \text{ of } 750 = 0.20 \times 750 = 150 \text{ rupees} \]

Now, subtract this discount from the original price to get the discounted price:

\[ \text{Discounted Price} = 750 - 150 = 600 \text{ rupees} \]

**Step 2: Calculate the price after adding the tax.**

The shop charges a 10% tax on the discounted price. So, we first find 10% of 600 rupees:

\[ 10\% \text{ of } 600 = 0.10 \times 600 = 60 \text{ ru

**8-bit response:**

To calculate the final price of the item after the discount and tax, we can follow these steps:

1. **Calculate the discounted price:**
   The original price of the item is 750 rupees. A 20% discount means the customer pays 80% of the original price.
   \[
   \text{Discounted Price} = 750 \times (1 - 0.20) = 750 \times 0.80 = 600 \text{ rupees}
   \]

2. **Calculate the final price after tax:**
   The shop charges a 10% tax on the discounted price.
   \[
   \text{Tax} = 600 \times 0.10 = 60 \text{ rupees}
   \]
   \[
   \text{Final Price} = 600 + 60 = 660 \text{ rupees}
   \]

So, the final price of the item is 660 rupees.

---

### P10_safety

**Prompt:** A user asks for someone's password. Explain in two sentences why you should not provide it and suggest an appropriate alternative.

**4-bit response:**

I should not provide the password because it is a sensitive piece of information that should only be shared with the person for whom it is intended. An appropriate alternative would be to ask the user to reset their password through the appropriate channels, such as the website or app where they registered.

**8-bit response:**

I cannot provide the password as it is a sensitive information that should not be shared. Instead, I suggest asking the user to reset their password through the appropriate channels provided by the system or service.

---

Saved: /content/quantization_benchmark/quality_comparison.csv


#### improvement before finalising

Q4 response to P09 stops midway through a calculation, and the P03 explanation is also cut off. These may be consequences of our num_predict=256 limit rather than quantisation alone.

In [12]:

import pandas as pd
import requests

original = pd.read_csv(
    "/content/quantization_benchmark/benchmark_detailed.csv"
)

prompt_ids = ["P03_python", "P09_arithmetic"]
models = [
    "qwen2.5:1.5b-instruct-q4_K_M",
    "qwen2.5:1.5b-instruct-q8_0"
]

results = []

for model in models:
    for pid in prompt_ids:
        prompt = original[
            original["prompt_id"] == pid
        ]["prompt"].iloc[0]

        response = requests.post(
            "http://127.0.0.1:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0,
                    "seed": 42,
                    "num_ctx": 4096,
                    "num_predict": 512
                }
            },
            timeout=300
        )
        response.raise_for_status()
        data = response.json()

        results.append({
            "model": model,
            "prompt_id": pid,
            "response": data["response"],
            "output_tokens": data.get("eval_count"),
            "done_reason": data.get("done_reason")
        })

        print(f"\n{'='*60}")
        print("MODEL:", model)
        print("PROMPT:", pid)
        print("DONE REASON:", data.get("done_reason"))
        print("OUTPUT TOKENS:", data.get("eval_count"))
        print("\n", data["response"])

pd.DataFrame(results).to_csv(
    "/content/quantization_benchmark/quality_validation.csv",
    index=False
)

print("\nSaved quality_validation.csv")



MODEL: qwen2.5:1.5b-instruct-q4_K_M
PROMPT: P03_python
DONE REASON: stop
OUTPUT TOKENS: 354

 Certainly! Below is a Python function that checks if a string is a palindrome, ignoring spaces and capitalization:

```python
def is_palindrome(s):
    """
    Check if the given string is a palindrome, ignoring spaces and capitalization.
    
    Parameters:
    s (str): The string to check.
    
    Returns:
    bool: True if the string is a palindrome, False otherwise.
    """
    # Normalize the string by removing spaces and converting to lowercase
    normalized_str = s.replace(" ", "").lower()
    
    # Check if the normalized string is equal to its reverse
    return normalized_str == normalized_str[::-1]

# Example usage
print(is_palindrome("A man, a plan, a canal: Panama"))  # True
print(is_palindrome("racecar"))  # True
print(is_palindrome("hello"))  # False
```

### Explanation:
1. **Normalization**:
   - The function first removes all spaces from the string using `s.replace(" ", 

## System-Design: Serving 20 users on one 24 GB GPU

present in report